# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Thallysson100/FlyRank_AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps the **Content Refresh Prioritization** lane into a formal Machine Learning task loop.

## 0. Setup (Colab or local)

In [7]:
import os, sys, subprocess
import pandas as pd, numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working dir: /home/thallysson/AreaDeTrabalho/FlyRank_AI
Starter data found. You're ready.


**The lane:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?[cite: 3]

## 1. My lane as an ML task (type)

**Task Type:** Binary Classification combined with Risk Scoring/Ranking.

**Why?**
The operational objective is to decide which content items require human review and editorial refresh. We frame this primarily as a **binary classification task** predicting whether a content page will experience a significant drop in organic performance (`is_declining_label = 1`) over a trailing 30-day window. 

By predicting the calibrated probability of decline $P(\text{declining} = 1 | \mathbf{X})$, we convert classification outputs into a **ranked priority queue**, ranking by engagement features such as impressions, clicks, scrolls, etc. Content strategists can then review pages ordered by highest engaged page with high probability of decline, enabling efficient allocation of editorial resources.

In [8]:
# Verify label creation rule: is_declining_label = (trend_direction == 'down')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Total Content Rows: {len(df):,}")
print(f"Task Class Distribution:\n{df['is_declining_label'].value_counts(normalize=True).round(4)}")
print(f"Unique Clients (Grouping unit): {df['client_id'].nunique()}")

Total Content Rows: 30,000
Task Class Distribution:
is_declining_label
1    0.5421
0    0.4579
Name: proportion, dtype: float64
Unique Clients (Grouping unit): 32


## 2. Target or proxy

**Target:** `is_declining_label` (Binary target: `1` if `trend_direction == 'down'`, `0` otherwise).

**Source & Origin:**
The target is an observed rule-based outcome derived from trailing metrics: `trend_direction == 'down'` occurs when the recent 30-day search impressions drop by more than 20% compared to the prior 30-day window (`trend_pct < -20.0%`).

**Critical Leakage Guardrail:**
According to the dataset rules, `trend_direction` and `trend_pct` are the mathematical sources of the label. Therefore, `trend_direction`, `trend_pct`, and 30-day target window metrics (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`) **must strictly never be used as model features**. Only historical baseline properties and 90-day activity rates up to prediction time are allowed.

In [9]:
# Verify target definition and enforce leakage prevention list
leakage_columns = [
    'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]

allowed_features = [col for col in df.columns if col not in leakage_columns and col not in ['content_id', 'client_id']]

print(f"\nNumber of safe feature candidates: {len(allowed_features)}")
print("Sample allowed features:", allowed_features[:5])


Number of safe feature candidates: 34
Sample allowed features: ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type']


## 3. Success metric

**Primary Metric:** **PR-AUC (Precision-Recall Area Under Curve)** and **Precision@K** (e.g., Precision@Top 50 pages per client).

**Defense of Metric:**
1. **Capacity Constraints & False Positive Cost:** Content marketing teams operate under strict operational constraints and can only refresh a fixed budget of $K$ pages per sprint (e.g., top 50 prioritized pages per client). In this decision workflow, false positives directly translate into wasted editorial hours spent reviewing healthy pages. Evaluating **Precision@K** aligns model assessment directly with the human execution capacity.
2. **Imbalance & Ranking Stability:** PR-AUC evaluates ranking performance across all threshold operating points focused strictly on the positive class (`is_declining_label = 1`), ignoring true negatives. This makes it more appropriate than ROC-AUC or Raw Accuracy for risk-scoring systems where prioritizing actual declining content at the top of the queue is what drives business value.
3. **Decision Baseline & Minimum Viable Threshold:** A random guess baseline yields a PR-AUC equal to the positive class prevalence (~54.2% in this slice). To be deployed, a model must achieve a meaningful lift over chance (e.g., $\text{PR-AUC} > \text{prevalence} + 0.15$) and deliver a **Precision@Top50 $\ge$ 0.80**, ensuring at least 80% of prioritized recommendations represent genuine refresh targets.

In [10]:
# Code only

## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** **One row = One single content item (page)** identified by `content_id`, aggregated over a trailing 90-day window.

Grouped by `client_id` (32 pseudonymized client portfolios) to ensure client-holdout validation splits.

In [11]:
df[allowed_features].head()

,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,...,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,...,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,...,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,...,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,...,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5


## 5. Why ML beats a fixed rule here

**Why Hard-coded If-Statements Fail:**
1. **Non-Linear Interactions:** Traffic decay is not driven by single isolated metrics. An article with low word count might perform well if it targets high-intent queries, whereas an authoritative long-form article with high impressions might decline due to subtle position losses or falling CTR. Fixed rules like `if age > 180 and position > 20` can create significant false-positive noise.
2. **Multi-Factor Tradeoffs:** ML models (e.g., Gradient Boosted Trees) automatically learn complex continuous interactions between keyword competition, engagement rates, search position tiers, and AI referral traffic percentages without manual heuristic thresholding.
3. **Heterogeneous Portfolios:** Client accounts (`client_id`) differ in baseline traffic scale and content types. ML models capture non-linear feature representations that generalize better across diverse client portfolios than brittle rules.

In [12]:
# Code demonstration: Evaluating a naive fixed rule vs actual labels
# Naive Rule: Flag page for refresh if content_age_days > 180 and ctr < 1.0
df['naive_rule_flag'] = ((df['content_age_days'] > 180) & (df['ctr'] < 1.0)).astype(int)

rule_precision = (df[df['naive_rule_flag'] == 1]['is_declining_label'] == 1).mean()
rule_recall = (df[df['is_declining_label'] == 1]['naive_rule_flag'] == 1).mean()
prevalence = df['is_declining_label'].mean()

print(f"Naive Fixed Rule Performance:")
print(f" - Precision: {rule_precision:.4f} (Baseline prevalence is {prevalence:.4f})")
print(f" - Recall:    {rule_recall:.4f}")
print("Conclusion: Fixed heuristics produce high false-positive rates and miss many declining pages")

Naive Fixed Rule Performance:
 - Precision: 0.4885 (Baseline prevalence is 0.5421)
 - Recall:    0.5019
Conclusion: Fixed heuristics produce high false-positive rates and miss many declining pages


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.